In [26]:
import os
import numpy as np
from csiread import Intel #Intel csı formatındaki .dat dosyalarını okumak için

In [27]:
file_path = "/Users/aleynagulkazdal/Desktop/Bitirme/WiAR-master/data/volunteer_a_all_data/csi_a10_12.dat"

csidata = Intel(file_path)
#.dat dosyasındaki CSI verileri belleğe yüklenir 
csidata.read()

#csidata.csi:
#Bu ham csı complex değerlerini içerir 
#Genelde shape: (T,30,3) olur
#T= zaman frame sayısı,30=subcarrier sayısı,3=anten sayısı
print("Rax CSI shape:",csidata.csi.shape)

Rax CSI shape:connector_log=0x1
194 0xbb packets parsed
 (194, 30, 3, 2)


In [28]:
#2 = Real + Imag (kompleks bileşen)
#Real +Imag -> Complex sayıya çevireceğiz

#Real ve Imag bileşenlerini ayırıyoruz 
real_part = csidata.csi[...,0]
imag_part = csidata.csi[...,1]

#complex sayı oluştur
csi_complex = real_part + 1j * imag_part

#Amplitude (mutlak) al
csi_amp = np.abs(csi_complex)

print("Amplitude shape: ", csi_amp.shape)



Amplitude shape:  (194, 30, 3)


In [29]:
# (293, 30, 3) --> (293,90)

csi_flat = csi_amp.reshape(csi_amp.shape[0],-1)

print ("Flattened shape :",csi_flat.shape)


Flattened shape : (194, 90)


In [ ]:
def fix_length(x, target=300):
    """
    x: (T, 90) shape'inde CSI amplitude verisi
    target: sabitlemek istediğimiz frame sayısı
    
    Eğer T > target:
        Fazla kısmı keseriz.
    Eğer T < target:
        Eksik kısmı sıfır padding ile doldururuz.
    """
    
    if x.shape[0] > target:
        # Fazla frame varsa kes
        return x[:target]
    
    elif x.shape[0] < target:
        # Eksik frame varsa sıfır ekle
        pad = np.zeros((target - x.shape[0], x.shape[1]))
        return np.vstack([x, pad])
    
    else:
        return x


x_fixed = fix_length(csi_flat)

print("Final shape:", x_fixed.shape)

Final shape: (300, 90)
